# 8. End-to-End Tourism Recommendation Pipeline

## Objective

Integrate the models and recommendation components developed in the previous notebooks into a single end-to-end tourism analytics and recommendation pipeline.

## Pipeline Components

- Rating Prediction using XGBoost Regression
- Visit Mode Prediction using XGBoost Classification
- User Segmentation using K-Means Clustering
- Personalized Recommendation using the Advanced Recommendation System

## Pipeline Flow

User / Historical Data
        ↓
Feature Preparation
        ↓
Rating Prediction
        ↓
Visit Mode Prediction
        ↓
User Segmentation
        ↓
Personalized Recommendation
        ↓
Final Tourism Recommendation Output

## Key Goals

- Load and reuse previously trained models
- Integrate the individual ML components
- Create a unified prediction and recommendation workflow
- Validate the pipeline using real users from the dataset
- Ensure consistent and reproducible outputs

In [2]:
# Load Dataset and Trained Model Artifacts

import pandas as pd
import numpy as np
import joblib

# Load engineered dataset
df = pd.read_csv("tourism_cleaned_engineered.csv")

# Load regression artifacts
regression_model = joblib.load("best_regression_model.pkl")
regression_feature_columns = joblib.load(
    "regression_feature_columns.pkl"
)
attraction_avg_rating_map = joblib.load(
    "attraction_avg_rating_map.pkl"
)

# Load classification artifacts
classification_model = joblib.load(
    "best_classification_model.pkl"
)
classification_feature_columns = joblib.load(
    "classification_feature_columns.pkl"
)
visitmode_label_map = joblib.load(
    "visitmode_label_map.pkl"
)

# Load clustering artifacts
clustering_model = joblib.load(
    "kmeans_clustering_model.pkl"
)
clustering_scaler = joblib.load(
    "clustering_scaler.pkl"
)
clustering_feature_columns = joblib.load(
    "clustering_feature_columns.pkl"
)
cluster_names = joblib.load(
    "cluster_names.pkl"
)

print("Dataset and model artifacts loaded successfully.")
print()
print("Dataset shape:", df.shape)
print("Regression features:", len(regression_feature_columns))
print("Classification features:", len(classification_feature_columns))
print("Clustering features:", clustering_feature_columns)
print("VisitMode labels:", visitmode_label_map)
print("Clusters:", cluster_names)

Dataset and model artifacts loaded successfully.

Dataset shape: (52930, 24)
Regression features: 159
Classification features: 157
Clustering features: ['total_visits', 'unique_attractions', 'avg_rating', 'unique_visit_modes']
VisitMode labels: {'Business': 0, 'Couples': 1, 'Family': 2, 'Friends': 3, 'Solo': 4}
Clusters: {0: 'Occasional High-Rating Visitors', 1: 'Multi-Mode Travelers', 2: 'Low-Satisfaction Visitors', 3: 'Frequent & Diverse Travelers', 4: 'Repeat Visitors'}


In [3]:
content_similarity_df = joblib.load("content_similarity_matrix.pkl")
attraction_lookup = joblib.load("attraction_lookup.pkl")

inv_mode_map = {v: k for k, v in visitmode_label_map.items()}
print("Recommendation artifacts loaded.")

Recommendation artifacts loaded.


In [4]:
def prepare_features(feature_columns, raw_input: dict):
    """Saved feature_columns list ke against ek aligned row banata hai.
    Missing/dummy columns automatically 0 ho jaते hain."""
    return pd.DataFrame([{c: raw_input.get(c, 0) for c in feature_columns}])

In [5]:
def tourism_recommendation_pipeline(user_id, visit_year=2023, visit_month=6):
    user_data = df[df['UserId'] == user_id]
    if user_data.empty:
        return "User not found."

    target_attraction_id = user_data['AttractionId'].iloc[0]
    country = user_data['Country'].iloc[0]
    attraction_type_id = df[df['AttractionId']==target_attraction_id]['AttractionTypeId'].iloc[0]
    attr_avg = attraction_avg_rating_map.get(target_attraction_id, df['Rating'].mean())
    user_visits = user_data.shape[0]

    # 1. Rating Prediction
    reg_input = {'VisitYear': visit_year, 'VisitMonth': visit_month,
                 'AttractionTypeId': attraction_type_id, 'user_visit_count': user_visits,
                 'attraction_avg_rating': attr_avg, f'Country_{country}': 1}
    predicted_rating = regression_model.predict(prepare_features(regression_feature_columns, reg_input))[0]

    # 2. Visit Mode Prediction
    clf_input = {**reg_input, 'Rating': predicted_rating}
    predicted_mode_id = classification_model.predict(prepare_features(classification_feature_columns, clf_input))[0]
    predicted_mode = inv_mode_map[predicted_mode_id]

    # 3. User Segment
    seg_features = pd.DataFrame([{
        'total_visits': user_visits,
        'unique_attractions': user_data['AttractionId'].nunique(),
        'avg_rating': user_data['Rating'].mean(),
        'unique_visit_modes': user_data['VisitMode'].nunique()
    }])[clustering_feature_columns]
    cluster_id = clustering_model.predict(clustering_scaler.transform(seg_features))[0]
    segment_name = cluster_names[cluster_id]

    # 4. Recommendations (content-based — works even for never-rated attractions)
    visited_ids = set(user_data['AttractionId'])
    scores = content_similarity_df[target_attraction_id].drop(labels=visited_ids, errors='ignore')
    top_ids = scores.sort_values(ascending=False).head(5).index
    recs = attraction_lookup[attraction_lookup['AttractionId'].isin(top_ids)]

    return {'user_id': user_id, 'predicted_rating': round(float(predicted_rating), 2),
            'predicted_visit_mode': predicted_mode, 'user_segment': segment_name,
            'recommended_attractions': recs['Attraction'].tolist()}

# Test on a real user
test_user = df['UserId'].value_counts().index[0]
result = tourism_recommendation_pipeline(test_user)
for k, v in result.items():
    print(f"{k}: {v}")

user_id: 60799
predicted_rating: 4.31
predicted_visit_mode: Family
user_segment: Frequent & Diverse Travelers
recommended_attractions: ['Tanah Lot Temple', 'Tegenungan Waterfall', 'Mount Semeru Volcano', 'Sempu Island', 'Botanical Garden - Sanaa']
